In [ ]:
version = '7.0.0'

In [ ]:
import json
import pandas as pd
import re

from scrape_builds import scrape_builds

In [ ]:
def normalize_df(df, col_name):
    normalized_data = pd.json_normalize(df[col_name]).add_prefix(f'{col_name}.')
    df = pd.concat([df, normalized_data], axis=1)
    df = df.drop(col_name, axis=1)
    return df

In [ ]:
scrape_builds(version)

In [ ]:
file_path = f'builds_output_{version}.json'
json_file = open(file_path)
json_dict = json.load(json_file)

df = pd.DataFrame.from_dict(json_dict)
df

In [ ]:
for item in ['artifacts', 'main_stats', 'substats']:
    df = normalize_df(df, item)

artifact_cols = [col_name for col_name in df.columns if 'artifacts' in col_name]
substat_cols = [col_name for col_name in df.columns if 'substats' in col_name]

# Add new columns for spreadsheet calculations
df['substats'] = df[substat_cols].apply(lambda row: ', '.join([x for x in row if pd.notna(x)]), axis=1)

df['artifacts_concat'] = '=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), COLUMN()- 4)) = 0), "", TEXTJOIN(", ", TRUE, INDIRECT(ADDRESS(ROW(), COLUMN() + 1)):INDIRECT(ADDRESS(ROW(), COLUMN() + START!$B$4))))'
df['substats_if'] = '=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), COLUMN()- 14)) = 0), "", INDIRECT(ADDRESS(ROW(), COLUMN()+1)))'

# Reorder columns
new_col_order = ['character_name', 'element', 'build_name', 'is_priority_build', 'main_stats.Sands', 'main_stats.Goblet', 'main_stats.Circlet', 'artifacts_concat']
new_col_order.extend(artifact_cols)
new_col_order.append('substats_if')
new_col_order.append('substats')
new_col_order.extend(substat_cols)

df = df[new_col_order]
df

In [ ]:
# Export final csv
df.to_csv(f'artifact_salvaging_helper_{version}.csv', index=False)
print(f'Output saved to artifact_salvaging_helper_{version}.csv')